# 01 - Data Loading, Dataset Structure & Landmark Feature Representation

## Overview
This notebook presents the data foundation and spatial feature extraction pipeline for our ASL fingerspelling recognition project.

We will explore:
1. Loading the saved dataset archives (`training_data.npz` and `test_data.npz`).
2. Examining class distributions and dataset shapes.
3. Understanding the 63-dimensional MediaPipe hand landmark representation.
4. Analyzing the mathematical normalization steps (wrist translation and 2D scale).
5. Discussing spatial invariances provided by normalization and critical limitations (e.g., rotation sensitivity).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

# Set up clean plotting style
plt.style.use('ggplot')
%matplotlib inline

## 1. Dataset Loading & `.npz` Structure

The dataset is stored in NumPy compressed archive format (`.npz`). Each file contains two key arrays:
- `X`: A 2D NumPy array of shape `(N, 63)` containing float32 landmark feature vectors.
- `y`: A 1D NumPy array of shape `(N,)` containing string class labels (e.g., `'A'`, `'F'`, `'SPACE'`).

We load datasets relative to the repository root using `../training_data.npz` and `../test_data.npz`.

In [ ]:
train_path = Path("../training_data.npz")
test_path = Path("../test_data.npz")

train_data = np.load(train_path)
test_data = np.load(test_path)

X_train, y_train = train_data["X"], train_data["y"]
X_test, y_test = test_data["X"], test_data["y"]

print(f"Training feature matrix X_train shape: {X_train.shape}")
print(f"Training label vector y_train shape:   {y_train.shape}")
print(f"Test feature matrix X_test shape:      {X_test.shape}")
print(f"Test label vector y_test shape:        {y_test.shape}")

## 2. Class Distribution Analysis

Let's examine the target classes present in both datasets. 

The dataset currently covers 12 target classes:
- **ASL Alphabetic Signs**: `A`, `D`, `F`, `I`, `L`, `N`, `O`, `T`, `U`
- **Control Gestures**: `SPACE`, `BACKSPACE`, `CLEAR`

In [ ]:
train_classes, train_counts = np.unique(y_train, return_counts=True)
test_classes, test_counts = np.unique(y_test, return_counts=True)

print("Class Distribution Breakdown:")
print(f"{'Class Label':<15} {'Train Samples':<15} {'Test Samples':<15} {'Total Samples':<15}")
print("-" * 60)

for label in sorted(train_classes):
    tr_cnt = int(train_counts[np.where(train_classes == label)[0][0]])
    te_cnt = int(test_counts[np.where(test_classes == label)[0][0]]) if label in test_classes else 0
    print(f"{label:<15} {tr_cnt:<15} {te_cnt:<15} {tr_cnt + te_cnt:<15}")

print("-" * 60)
print(f"{'TOTAL':<15} {len(y_train):<15} {len(y_test):<15} {len(y_train) + len(y_test):<15}")

In [ ]:
# Visualize Class Distributions
fig, ax = plt.subplots(figsize=(10, 5))
x_indices = np.arange(len(train_classes))
width = 0.35

rects1 = ax.bar(x_indices - width/2, train_counts, width, label='Train (n=199)', color='#3498db')
rects2 = ax.bar(x_indices + width/2, test_counts, width, label='Test (n=123)', color='#e74c3c')

ax.set_ylabel('Sample Count')
ax.set_title('Dataset Sample Counts per Class (Train vs Held-Out Test)')
ax.set_xticks(x_indices)
ax.set_xticklabels(train_classes, rotation=45, ha='right')
ax.legend()
plt.tight_layout()
plt.show()

## 3. MediaPipe 63-Dimensional Feature Representation

MediaPipe Hand Landmarker tracks **21 key points** on a single hand. Each landmark is represented by 3 spatial coordinates:
- $x \in [0, 1]$: Horizontal normalized image coordinate.
- $y \in [0, 1]$: Vertical normalized image coordinate.
- $z$: Relative depth coordinate (origin at wrist).

$$21 	ext{ landmarks} 	imes 3 	ext{ coordinates } (x, y, z) = 63 	ext{ features}$$

### The 21 Hand Landmarks:
- **0**: Wrist
- **1-4**: Thumb (CMC, MCP, IP, Tip)
- **5-8**: Index Finger (MCP, PIP, DIP, Tip)
- **9-12**: Middle Finger (MCP, PIP, DIP, Tip)
- **13-16**: Ring Finger (MCP, PIP, DIP, Tip)
- **17-20**: Pinky Finger (MCP, PIP, DIP, Tip)

In [ ]:
sample_features = X_train[0]
sample_landmarks = sample_features.reshape(21, 3)

print(f"Flat feature vector shape: {sample_features.shape}")
print(f"Reshaped 3D landmark matrix shape: {sample_landmarks.shape}")
print("
First 5 Landmarks (x, y, z):")
for i in range(5):
    print(f"Landmark {i}: x={sample_landmarks[i,0]:.4f}, y={sample_landmarks[i,1]:.4f}, z={sample_landmarks[i,2]:.4f}")

## 4. Mathematical Normalization Pipeline

Raw landmark coordinates depend directly on where the hand is located in the video frame and how far the hand is from the camera lens.

To make features comparable across frames, the project applies two normalization transformations in `sign_type.py`:

### 1. Wrist Translation Normalization (Origin Shift)
The wrist landmark (index `0`) is subtracted from all 21 landmark points:

$$\mathbf{p}_i' = \mathbf{p}_i - \mathbf{p}_0 \quad 	ext{for } i \in \{0, \dots, 20\}$$

This forces the wrist to always sit at coordinate origin $(0, 0, 0)$.

### 2. Scale Normalization (Distance Scaling)
We compute the maximum 2D Euclidean distance from the wrist origin to any landmark in the $xy$-plane:

$$s = \max_{i \in \{0, \dots, 20\}} \sqrt{(x_i')^2 + (y_i')^2}$$

All 3D coordinates are divided by $s$:

$$\mathbf{p}_i'' = rac{\mathbf{p}_i'}{s}$$

This rescales the hand bounding radius in the $xy$-plane to $1.0$.

In [ ]:
# Illustrate normalization step by step on a dummy raw landmark array
np.random.seed(42)
raw_points = np.random.uniform(low=0.2, high=0.8, size=(21, 3)).astype(np.float32)
# Simulate wrist at (0.5, 0.6, 0.1)
raw_points[0] = [0.5, 0.6, 0.1]

# Step 1: Wrist Origin Translation
points_translated = raw_points - raw_points[0]

# Step 2: Scale Normalization
scale = np.max(np.linalg.norm(points_translated[:, :2], axis=1))
points_normalized = points_translated / scale if scale > 0 else points_translated

print(f"Raw Wrist Landmark (0): {raw_points[0]}")
print(f"Translated Wrist (0):  {points_translated[0]}")
print(f"Max 2D Scale Factor s: {scale:.4f}")
print(f"Normalized Wrist (0):  {points_normalized[0]}")
print(f"Max 2D Norm after scaling: {np.max(np.linalg.norm(points_normalized[:, :2], axis=1)):.4f}")

## 5. Invariance Discussion

### Invariances Provided:
1. **Translation Invariance**: Moving the hand anywhere across the image frame shifts coordinates uniformly. Wrist subtraction eliminates frame-position dependency.
2. **2D Scale / Distance Invariance**: Moving the hand closer or further from the camera changes the apparent pixel size. Scale division normalizes hands of varying physical distances to a consistent bounding scale.

### What Normalization Does NOT Provide:
1. **Rotation Invariance (Roll / Pitch / Yaw)**: 
   - Rotating the hand in 2D (tilted wrist) or 3D space rotates all landmark vectors relative to coordinate axes. 
   - The current pipeline does **not** align the hand along a canonical axis (e.g., aligning the wrist-to-middle-finger vector along the vertical $y$-axis).
   - As a result, a sign performed with a tilted hand produces a substantially different 63D vector.
2. **Signer Anatomy Invariances**:
   - Differences in finger length ratios, palm size vs. finger length, or joint flexibility across different signers alter relative landmark positions.
3. **Perspective Distortion Invariances**:
   - Non-linear camera lens perspective effects at frame edges are not corrected by linear scaling.

## Key Takeaways & Transition
- We have verified that `training_data.npz` contains 199 samples and `test_data.npz` contains 123 samples across 12 target classes.
- Feature vectors consist of 63 values derived from 21 MediaPipe 3D hand landmarks.
- Wrist-origin subtraction and max-distance scaling provide translation and scale invariance.
- Lack of rotation normalization means models must either rely on consistent signing posture or data augmentation.
- In `02_knn_baseline.ipynb`, we evaluate how well K-Nearest Neighbors classifies these 63-dimensional feature vectors.